In [4]:
import torch
import torch.nn as nn
from transformer import Linear

# 问题 8：实现逐位置前馈网络（2 分）
class FFN(torch.nn.Module):
    def __init__(self, d_model: int, d_ff: int=512, device=None, dtype=None):
        super().__init__()
        '''
        d_model: int  模型隐藏层/词向量的维度
        d_ff: int  前馈网络中间层维度
        device: torch.device | None = None  参数存放设备
        dtype: torch.dtype | None = None  参数的数据类型
        '''
        # 此处编写你的参数创建代码
        # 127 // 64 = 1,  1 * 64 = 64,
        self.d_ff = 64 * (8 * d_model // 3 // 64) # // 向下整除符号；d_ff 一般取 8/3 * d_model 最近的 64 的倍数
        self.w1, self.w3 = Linear(d_model, self.d_ff, device=device, dtype=dtype), Linear(d_model, self.d_ff, device=device, dtype=dtype)
        self.w2 = Linear(self.d_ff, d_model, device=device, dtype=dtype)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        '''
        FFN(x) = W2(SiLU(W1*x) ⊙ W3*x)
        ⊙ 代表逐‑元素相乘
        x: (batch_size, seq_len, d_model)
        return: (batch_size, seq_len, d_model)
        '''
        y1 = self.w1(x)                          # (B, S, d_model) -> (B, S, d_ff)  W1 线性映射
        # SiLU(x) = x * sigmoid(x)，用 torch.sigmoid 保证数值稳定；再与 W3 分支逐元素相乘形成 GLU 门控
        y13 = y1 * torch.sigmoid(y1) * self.w3(x) # (B, S, d_ff) ⊙ (B, S, d_ff) -> (B, S, d_ff)
        return self.w2(y13)                      # (B, S, d_ff) -> (B, S, d_model)  输出投影


In [5]:
# 测试 FFN（SwiGLU 逐位置前馈网络）
batch_size, seq_len, d_model = 2, 5, 128
d_ff = 512

ffn = FFN(d_model=d_model, d_ff=d_ff)
x_in = torch.randn(batch_size, seq_len, d_model)
ffn_out = ffn(x_in)
print(f"FFN 输入形状：{x_in.shape}，FFN 输出形状：{ffn_out.shape}")


FFN 输入形状：torch.Size([2, 5, 128])，FFN 输出形状：torch.Size([2, 5, 128])
